[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/07-integrations/03-pandas_etl_pipeline.ipynb)

In [1]:
# !pip install mbox pandas

# M|BOX as a Stage in a Pandas ETL Pipeline

`06-agentic-ai/08` matched an entire batch in one call and routed the ambiguous rows to an LLM. This notebook is a different, more common shape of problem: a plain data pipeline, extract, transform, load, with no LLM anywhere in it, where the transform step needs to check an incoming file against a table you already trust before any of it lands in your warehouse. This is dedup-on-ingest, or record linkage against a reference table, matching new rows onto an existing, already-trusted master. Every incoming row is checked against that one master table and nothing else, there's no grouping of incoming rows against each other going on here. It also covers the part that matters once a file stops being small enough to eyeball: processing it in chunks, without rebuilding the index for every chunk.

In this notebook you will:

1. Set up a pipeline's shape: a trusted master table, and a raw incoming file that hasn't been checked against it yet
2. Build the M|BOX index once, outside the processing loop, the way an ETL stage actually should
3. Write the transform step as a plain function that labels every incoming row with its match, not just the ones that cross a threshold
4. Run that transform over the incoming file in chunks, and confirm the result is identical to processing it all at once
5. Load the outcome, new rows staged for the warehouse, everything else kept in one auditable, labeled table

In [1]:
import pandas as pd

## 1. The pipeline's shape

`customer_master.csv` is the trusted table, 45 customers already in the warehouse. `daily_signups.csv` is a raw incoming file, 150 rows from today's signup form, not yet checked against anything. Some of these are genuinely new customers. Some are existing customers signing up again, typo'd, differently capitalized, whitespace mangled, the ordinary mess of a form nobody validated against the master table at entry time.

In [2]:
master = pd.read_csv("datasets/customer_master.csv")
signups = pd.read_csv("datasets/daily_signups.csv")
print(f"{len(master)} trusted customers, {len(signups)} incoming signups to check")
signups.head()

45 trusted customers, 150 incoming signups to check


,signup_name,signup_email
0,Richard Thompson,richard.thompson38@hotmail.com
1,linda perez,signup+9834@yahoo.com
2,Matthew Martinez,matthew.martinez79@hotmail.com
3,John Wilson,john.wilson24@outlook.com
4,Barbara Lee,barbara.lee66@icloud.com


## 2. Build the index once, outside the loop

This is the one M|BOX-specific step, and it happens exactly once, before any chunk of the incoming file is touched. Every chunk below reuses this same `index` object.

In [3]:
from mbox.indexing import TableIndexer
from mbox.recall import TableRecallConfig, TableRecallFieldConfig, TableRecallMode

index = TableIndexer.create_index(master, index_columns=["full_name"], tmp_dir="tmp_index_master")
config = TableRecallConfig(
    fields=[TableRecallFieldConfig(input_column="full_name", indexed_column="full_name",
                                    minimum_quality=0, weight=100, mode=TableRecallMode.APPROX)],
    max_results=1, min_total_match_value=0, include_field_scores=True
)
DUPLICATE_THRESHOLD = 70

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


## 3. The transform step, as a plain function

A real pipeline stage is a function with a clear input and output, not a one-off script. This one takes a chunk of incoming rows and the already-built index, and returns that same chunk with three columns added: the closest master record found, its score, and whether that score clears the duplicate threshold. Every row keeps its match information, not just the ones that end up flagged, that matters in the next section.

In [4]:
def label_signup_chunk(chunk: pd.DataFrame, index, config) -> pd.DataFrame:
    queries = chunk.rename(columns={"signup_name": "full_name"})[["full_name"]]
    matches = index.match(queries=queries, config=config)

    labeled = chunk.copy()
    labeled["matched_master_name"] = matches["full_name_candidate"].values
    labeled["match_score"] = matches["full_name_score"].values
    labeled["is_duplicate"] = labeled["match_score"] >= DUPLICATE_THRESHOLD
    return labeled

## 4. Run it over the file in chunks

`pd.read_csv(..., chunksize=N)` yields the file 50 rows at a time instead of loading all 150 in memory at once, this is the part of the pattern that actually matters once "the file" stops being something you can comfortably eyeball. The index built in section 2 is reused for every chunk, never rebuilt, and every chunk's labeled result is collected into one running list.

In [5]:
labeled_batches = []

for i, chunk in enumerate(pd.read_csv("datasets/daily_signups.csv", chunksize=50)):
    labeled_chunk = label_signup_chunk(chunk, index, config)
    n_duplicate = int(labeled_chunk["is_duplicate"].sum())
    print(f"chunk {i}: {len(chunk)} rows in, {n_duplicate} flagged as duplicate, {len(chunk) - n_duplicate} new")
    labeled_batches.append(labeled_chunk)

labeled_signups = pd.concat(labeled_batches, ignore_index=True)
print(f"\nTotal: {(~labeled_signups['is_duplicate']).sum()} new, "
      f"{labeled_signups['is_duplicate'].sum()} duplicate, out of {len(labeled_signups)} incoming rows")

chunk 0: 50 rows in, 8 flagged as duplicate, 42 new
chunk 1: 50 rows in, 9 flagged as duplicate, 41 new
chunk 2: 50 rows in, 13 flagged as duplicate, 37 new

Total: 120 new, 30 duplicate, out of 150 incoming rows


Chunking changed nothing about the outcome, each row is still matched against the full 45-row master index regardless of which chunk it landed in, chunk boundaries are purely a memory and throughput decision, not an accuracy one. `120` new plus `30` duplicate should account for all `150` incoming rows.

One table, every row labeled, nothing split off into a second variable yet.

In [12]:
labeled_signups

,signup_name,signup_email,matched_master_name,match_score,is_duplicate
0,Richard Thompson,richard.thompson38@hotmail.com,,0,False
1,linda perez,signup+9834@yahoo.com,Linda Perez,100,True
2,Matthew Martinez,matthew.martinez79@hotmail.com,,0,False
3,John Wilson,john.wilson24@outlook.com,,0,False
4,Barbara Lee,barbara.lee66@icloud.com,,0,False
...,...,...,...,...,...
145,Charles Martin,charles.martin6@hotmail.com,,0,False
146,Sandra Robinson,sandra.robinson31@yahoo.com,,0,False
147,Susan Lopez,susan.lopez94@outlook.com,,0,False
148,James Perez,james.perez71@outlook.com,,0,False


Let's checkout the duplicated rows

In [14]:
labeled_signups[labeled_signups.is_duplicate]

,signup_name,signup_email,matched_master_name,match_score,is_duplicate
1,linda perez,signup+9834@yahoo.com,Linda Perez,100,True
12,Patrciia Robinson,signup+2104@gmail.com,Patricia Robinson,87,True
15,Robert Jackson,signup+8397@gmail.com,Robert Jackson,100,True
22,David Williams,signup+4258@yahoo.com,David Williams,100,True
24,Robert Gonzalez,signup+1188@gmail.com,Robert Gonzalez,100,True
36,Kaern Lee,signup+2290@gmail.com,Karen Lee,70,True
41,Linda Jones,signup+2160@hotmail.com,Linda Jones,100,True
46,Sandra Jackso,signup+7669@yahoo.com,Sandra Jackson,83,True
56,Mihcael Lee,signup+8062@outlook.com,Michael Lee,78,True
64,david jones,signup+1117@gmail.com,David Jones,100,True


## 5. Why keep the score on rows that didn't cross the threshold

Splitting into a "new" DataFrame and a "duplicate" DataFrame the moment a threshold is applied throws away the one piece of information that would let you check the threshold's judgment later, how close every other row actually came. Sort the rows that were *not* flagged as duplicates by their score anyway.

In [8]:
labeled_signups[~labeled_signups["is_duplicate"]].sort_values("match_score", ascending=False).head(5)

,signup_name,signup_email,matched_master_name,match_score,is_duplicate
70,David Johnson,david.johnson14@hotmail.com,David Robinson,48,False
0,Richard Thompson,richard.thompson38@hotmail.com,,0,False
3,John Wilson,john.wilson24@outlook.com,,0,False
2,Matthew Martinez,matthew.martinez79@hotmail.com,,0,False
5,Karen Clark,karen.clark93@hotmail.com,,0,False


`David Johnson` scored `48` against `David Robinson`, correctly kept as a new customer, matching last names would be a real coincidence worth ignoring, but that score would have been invisible in a design that only recorded match information for rows that got flagged. Keeping it on every row is what makes `DUPLICATE_THRESHOLD` an inspectable, adjustable decision instead of a silent cutoff, the same reasoning `06-agentic-ai/08` applied by keeping a `bucket` column on one summary table instead of splitting into three.

## 6. Load: derive the two outcomes from the one labeled table

Only now, right before the actual load step, does the single labeled table get filtered into two views. New customers are staged for the warehouse load, a real pipeline would `INSERT` these into the actual master table here, this notebook writes them to a CSV instead so re-running it doesn't quietly grow `customer_master.csv` on every execution. The full labeled table, duplicates included, is kept as the audit log, so a human can spot-check the threshold's calls, including the near misses, rather than trusting them silently.

In [9]:
new_customers = labeled_signups.loc[~labeled_signups["is_duplicate"], ["signup_name", "signup_email"]]
new_customers.to_csv("datasets/staged_new_customers.csv", index=False)
labeled_signups.to_csv("datasets/labeled_signups_log.csv", index=False)

print(f"Staged {len(new_customers)} new customers for load, "
      f"logged all {len(labeled_signups)} rows with scores for review.")

Staged 120 new customers for load, logged all 150 rows with scores for review.


## 7. Practical notes

**Build the index once, before the chunk loop, not inside it.** Rebuilding a 45-row index 3 times costs little here, rebuilding a real master index once per chunk, on a table with millions of rows and hundreds of chunks, turns a cheap pipeline stage into the slowest part of the run for no benefit, the master table isn't changing between chunks.

**Chunk size is a memory and throughput knob, not a correctness knob.** Every row in every chunk is still matched against the complete master index. Picking a smaller or larger `chunksize` changes how much of the incoming file sits in memory at once and how often progress gets logged, it does not change which rows come back as duplicates.

**Label every row, then filter, don't filter first and lose the rest.** A `DUPLICATE_THRESHOLD` is a judgment call, the same "thresholds are a decision, not a formality" point `06-agentic-ai/06` made about confidence-based escalation applies here too. Keeping `matched_master_name` and `match_score` on every row, not just the ones that crossed the line, is what lets you actually audit that judgment call later, including the near misses that almost did.

**Don't mutate the table you're checking against, mid-pipeline.** This notebook writes new customers to a separate staging file specifically so re-running the notebook reproduces the same result every time. A real load step appends to the actual master table exactly once, as a deliberate, separate action, not as a side effect of re-executing a notebook cell.

## Next steps

- **`01-fastapi_match_service.ipynb`** - expose this same kind of resolution as an HTTP endpoint other systems can call directly
- **`02-sql_database_integration.ipynb`** - source the master table this pipeline checks against from a real database